# Phase 10 — MLflow Tracking
## Real Estate Investment Advisor

This notebook tracks the **already-trained** classification and regression models.

### Classification
- Logistic Regression
- Decision Tree
- Random Forest
- Extra Trees
- XGBoost

### Regression
- Linear Regression
- Decision Tree
- Random Forest
- Extra Trees
- XGBoost

**Important:** This version uses a SQLite MLflow backend instead of the deprecated filesystem tracking backend (`mlruns`).

In [1]:
import os
import joblib
import pandas as pd
import mlflow
import mlflow.sklearn

print("MLflow version:", mlflow.__version__)
print("Python version:", __import__("sys").version.split()[0])

MLflow version: 3.16.1
Python version: 3.12.10


## 1. Configure MLflow with SQLite

In [2]:
# Project root is one level above the notebooks folder
PROJECT_ROOT = os.path.abspath("..")

# SQLite database for MLflow experiment/run metadata
MLFLOW_DB = os.path.join(PROJECT_ROOT, "mlflow.db")

# Local folder for MLflow model artifacts
MLFLOW_ARTIFACTS = os.path.join(PROJECT_ROOT, "mlartifacts")
os.makedirs(MLFLOW_ARTIFACTS, exist_ok=True)

# IMPORTANT:
# Use SQLite for MLflow tracking metadata.
# Do NOT use ../mlruns as the tracking URI.
tracking_uri = "sqlite:///" + MLFLOW_DB.replace("\\", "/")

mlflow.set_tracking_uri(tracking_uri)

print("MLflow tracking URI:")
print(mlflow.get_tracking_uri())

print("\nMLflow database:")
print(MLFLOW_DB)

print("\nMLflow artifact directory:")
print(MLFLOW_ARTIFACTS)

MLflow tracking URI:
sqlite:///d:/guvi/Real Estate Investment Advisor/mlflow.db

MLflow database:
d:\guvi\Real Estate Investment Advisor\mlflow.db

MLflow artifact directory:
d:\guvi\Real Estate Investment Advisor\mlartifacts


In [3]:
# Initialize the MLflow client after setting the tracking URI

from mlflow.tracking import MlflowClient

client = MlflowClient()

print("MLflow SQLite backend initialized successfully.")

2026/09/25 19:10:31 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/25 19:10:31 INFO mlflow.store.db.utils: Updating database tables


MLflow SQLite backend initialized successfully.


## 2. Verify Required Model Files

In [4]:
required_files = [
    "../models/classification_results.csv",
    "../models/logistic_regression.pkl",
    "../models/decision_tree.pkl",
    "../models/random_forest.pkl",
    "../models/extra_trees.pkl",
    "../models/xgboost_classifier.pkl",
    "../models/regression_results.csv",
    "../models/linear_regression.pkl",
    "../models/decision_tree_regressor.pkl",
    "../models/random_forest_regressor.pkl",
    "../models/extra_trees_regressor.pkl",
    "../models/xgboost_regressor.pkl",
]

missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print("Missing required files:")
    for file in missing_files:
        print(" -", file)
    raise FileNotFoundError(
        "Required model/result files are missing. "
        "Make sure Phase 8 and Phase 9 were completed first."
    )

print("All required model and result files are present.")

All required model and result files are present.


## Part A — Classification

In [5]:
classification_results = pd.read_csv(
    "../models/classification_results.csv"
)

print("Classification results:")
display(classification_results)

Classification results:


,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.93460,0.936966,0.937613,0.937289,0.987433
1,Decision Tree,0.99998,0.999962,1.000000,0.999981,0.999979
2,Random Forest,0.99868,0.997550,0.999923,0.998735,0.999984
3,Extra Trees,0.92466,0.928571,0.926754,0.927662,0.980200
4,XGBoost,0.99978,1.000000,0.999578,0.999789,0.999998


In [6]:
# Load trained classification models

classification_models = {
    "Logistic Regression": joblib.load("../models/logistic_regression.pkl"),
    "Decision Tree": joblib.load("../models/decision_tree.pkl"),
    "Random Forest": joblib.load("../models/random_forest.pkl"),
    "Extra Trees": joblib.load("../models/extra_trees.pkl"),
    "XGBoost": joblib.load("../models/xgboost_classifier.pkl"),
}

print("Loaded classification models:")
for name, model in classification_models.items():
    print(f"- {name}: {type(model).__name__}")

Loaded classification models:
- Logistic Regression: LogisticRegression
- Decision Tree: DecisionTreeClassifier
- Random Forest: RandomForestClassifier
- Extra Trees: ExtraTreesClassifier
- XGBoost: XGBClassifier


In [8]:
# Create or retrieve the classification experiment

classification_experiment_name = (
    "Real Estate Investment Advisor - Classification"
)

classification_experiment = mlflow.get_experiment_by_name(
    classification_experiment_name
)

if classification_experiment is None:
    classification_experiment_id = mlflow.create_experiment(
        classification_experiment_name
    )
else:
    classification_experiment_id = classification_experiment.experiment_id

print("Classification experiment ID:", classification_experiment_id)

Classification experiment ID: 1


In [10]:
# Log classification models and metrics

SKOPS_TRUSTED_TYPES = [
    "sklearn.tree._tree.Tree"
]

for model_name, model in classification_models.items():

    matching_rows = classification_results[
        classification_results["Model"].astype(str).str.strip() == model_name
    ]

    if matching_rows.empty:
        raise ValueError(
            f"No results row found for classification model: {model_name}"
        )

    result = matching_rows.iloc[0]

    with mlflow.start_run(
        experiment_id=classification_experiment_id,
        run_name=model_name
    ):

        mlflow.log_param("model_type", model_name)

        mlflow.log_metric("accuracy", float(result["Accuracy"]))
        mlflow.log_metric("precision", float(result["Precision"]))
        mlflow.log_metric("recall", float(result["Recall"]))
        mlflow.log_metric("f1_score", float(result["F1 Score"]))
        mlflow.log_metric("roc_auc", float(result["ROC AUC"]))

        if model_name == "XGBoost":
            import mlflow.xgboost

            mlflow.xgboost.log_model(
                model,
                name="model"
            )

        else:
            mlflow.sklearn.log_model(
                model,
                name="model",
                skops_trusted_types=SKOPS_TRUSTED_TYPES
            )

        print(f"Logged: {model_name}")

Logged: Logistic Regression
Logged: Decision Tree
Logged: Random Forest
Logged: Extra Trees
Logged: XGBoost


In [11]:
# Verify classification runs

classification_runs = mlflow.search_runs(
    experiment_ids=[classification_experiment_id],
    order_by=["start_time DESC"]
)

classification_columns = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1_score",
    "metrics.roc_auc"
]

display(classification_runs[classification_columns])

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc
0,176e590b0286459d8d32e52de7129726,XGBoost,0.99978,1.000000,0.999578,0.999789,0.999998
1,dc695a1b26124dcc8270401afa5e0ce2,Extra Trees,0.92466,0.928571,0.926754,0.927662,0.980200
2,1e01b2c96f8e49069027418081733e6c,Random Forest,0.99868,0.997550,0.999923,0.998735,0.999984
3,36eb39e5ee474b84af02b188233d695c,Decision Tree,0.99998,0.999962,1.000000,0.999981,0.999979
4,65970579710841b5bd8e86dca1352b04,Logistic Regression,0.93460,0.936966,0.937613,0.937289,0.987433
5,d45f61a822fa4ebf8eb4fe333566fec0,Decision Tree,0.99998,0.999962,1.000000,0.999981,0.999979
6,a990b63ec1dc4d48a1b64307c213b9d6,Logistic Regression,0.93460,0.936966,0.937613,0.937289,0.987433


## Part B — Regression

In [12]:
regression_results = pd.read_csv(
    "../models/regression_results.csv"
)

print("Regression results:")
display(regression_results)

Regression results:


,Model,MAE,MSE,RMSE,R2
0,Linear Regression,0.000900,0.000001,0.001222,1.000000
1,Decision Tree,0.008105,0.000110,0.010489,1.000000
2,Random Forest,0.002707,0.000015,0.003934,1.000000
3,Extra Trees,0.079119,0.011147,0.105580,1.000000
4,XGBoost,0.846246,1.109239,1.053204,0.999974


In [13]:
# Load trained regression models

regression_models = {
    "Linear Regression": joblib.load("../models/linear_regression.pkl"),
    "Decision Tree": joblib.load("../models/decision_tree_regressor.pkl"),
    "Random Forest": joblib.load("../models/random_forest_regressor.pkl"),
    "Extra Trees": joblib.load("../models/extra_trees_regressor.pkl"),
    "XGBoost": joblib.load("../models/xgboost_regressor.pkl"),
}

print("Loaded regression models:")
for name, model in regression_models.items():
    print(f"- {name}: {type(model).__name__}")

Loaded regression models:
- Linear Regression: LinearRegression
- Decision Tree: DecisionTreeRegressor
- Random Forest: RandomForestRegressor
- Extra Trees: ExtraTreesRegressor
- XGBoost: XGBRegressor


In [15]:
# Create or retrieve the regression experiment

regression_experiment_name = (
    "Real Estate Investment Advisor - Regression"
)

regression_experiment = mlflow.get_experiment_by_name(
    regression_experiment_name
)

if regression_experiment is None:
    regression_experiment_id = mlflow.create_experiment(
        regression_experiment_name
    )
else:
    regression_experiment_id = regression_experiment.experiment_id

print("Regression experiment ID:", regression_experiment_id)

Regression experiment ID: 2


In [17]:
# Log regression models and metrics

SKOPS_TRUSTED_TYPES = [
    "sklearn.tree._tree.Tree"
]

for model_name, model in regression_models.items():

    matching_rows = regression_results[
        regression_results["Model"].astype(str).str.strip() == model_name
    ]

    if matching_rows.empty:
        raise ValueError(
            f"No results row found for regression model: {model_name}"
        )

    result = matching_rows.iloc[0]

    with mlflow.start_run(
        experiment_id=regression_experiment_id,
        run_name=model_name
    ):

        mlflow.log_param("model_type", model_name)

        mlflow.log_metric("MAE", float(result["MAE"]))
        mlflow.log_metric("MSE", float(result["MSE"]))
        mlflow.log_metric("RMSE", float(result["RMSE"]))
        mlflow.log_metric("R2", float(result["R2"]))

        if model_name == "XGBoost":
            import mlflow.xgboost

            mlflow.xgboost.log_model(
                model,
                name="model"
            )

        else:
            mlflow.sklearn.log_model(
                model,
                name="model",
                skops_trusted_types=SKOPS_TRUSTED_TYPES
            )

        print(f"Logged: {model_name}")

Logged: Linear Regression
Logged: Decision Tree
Logged: Random Forest
Logged: Extra Trees
Logged: XGBoost


In [18]:
# Verify regression runs

regression_runs = mlflow.search_runs(
    experiment_ids=[regression_experiment_id],
    order_by=["start_time DESC"]
)

regression_columns = [
    "run_id",
    "tags.mlflow.runName",
    "metrics.MAE",
    "metrics.MSE",
    "metrics.RMSE",
    "metrics.R2"
]

display(regression_runs[regression_columns])

,run_id,tags.mlflow.runName,metrics.MAE,metrics.MSE,metrics.RMSE,metrics.R2
0,17ff9c205f1a4c4f9cf4041e121a4dcd,XGBoost,0.846246,1.109239,1.053204,0.999974
1,485bbdff64924786832008f064df3315,Extra Trees,0.079119,0.011147,0.105580,1.000000
2,70287de0fd984c91aa0465034b97f401,Random Forest,0.002707,0.000015,0.003934,1.000000
3,7c441709b673422b97b00555f3214eb5,Decision Tree,0.008105,0.000110,0.010489,1.000000
4,c2232a9bff414b9982b5877d4ee8c2b8,Linear Regression,0.000900,0.000001,0.001222,1.000000
5,11d4dcd2975f418ea70f48ff77f8fc98,Decision Tree,0.008105,0.000110,0.010489,1.000000
6,e15f0d089e504d82994c22cbcc74a5b8,Linear Regression,0.000900,0.000001,0.001222,1.000000


## Part C — Save MLflow Run Summaries

In [19]:
classification_mlflow_summary = classification_runs[
    classification_columns
].copy()

regression_mlflow_summary = regression_runs[
    regression_columns
].copy()

classification_mlflow_summary.to_csv(
    "../models/mlflow_classification_runs.csv",
    index=False
)

regression_mlflow_summary.to_csv(
    "../models/mlflow_regression_runs.csv",
    index=False
)

print("Saved:")
print("- ../models/mlflow_classification_runs.csv")
print("- ../models/mlflow_regression_runs.csv")

Saved:
- ../models/mlflow_classification_runs.csv
- ../models/mlflow_regression_runs.csv


In [20]:
# Final verification

print("=" * 65)
print("PHASE 10 — MLFLOW TRACKING COMPLETE")
print("=" * 65)

print("Tracking URI:")
print(mlflow.get_tracking_uri())

print("\nClassification experiment:")
print(classification_experiment_name)
print("Runs found:", len(classification_runs))

print("\nRegression experiment:")
print(regression_experiment_name)
print("Runs found:", len(regression_runs))

print("\nMLflow database:")
print(MLFLOW_DB)

print("\nMLflow artifacts:")
print(MLFLOW_ARTIFACTS)

print("\nSummary files:")
print("../models/mlflow_classification_runs.csv")
print("../models/mlflow_regression_runs.csv")

print("\nSUCCESS — Phase 10 MLflow tracking completed.")

PHASE 10 — MLFLOW TRACKING COMPLETE
Tracking URI:
sqlite:///d:/guvi/Real Estate Investment Advisor/mlflow.db

Classification experiment:
Real Estate Investment Advisor - Classification
Runs found: 7

Regression experiment:
Real Estate Investment Advisor - Regression
Runs found: 7

MLflow database:
d:\guvi\Real Estate Investment Advisor\mlflow.db

MLflow artifacts:
d:\guvi\Real Estate Investment Advisor\mlartifacts

Summary files:
../models/mlflow_classification_runs.csv
../models/mlflow_regression_runs.csv

SUCCESS — Phase 10 MLflow tracking completed.
